In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas
import numpy as np

import os


In [8]:
df = pd.read_csv('cleaned_sales.csv')

df.head(10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDateTime,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,15.30
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,25.50
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,11.10
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,11.10
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,54.08


In [9]:
INPUT_DIR = "."
OUTPUT_DIR = "."
CHART_DIR = os.path.join(OUTPUT_DIR, "charts")

cleaned_path = os.path.join(INPUT_DIR, "cleaned_sales.csv")
returns_path = os.path.join(INPUT_DIR, "returned.csv")

print("Looking in:", os.path.abspath(INPUT_DIR))
print("Files here:", os.listdir(INPUT_DIR))

if not os.path.exists(cleaned_path):
    raise FileNotFoundError(
        f"File not found: '{cleaned_path}', Current dir:{os.getcwd()}."
        f"Files present: {os.listdir('.')}"
        )
    
os.makedirs(CHART_DIR, exist_ok=True)
sns.set_theme

df = pd.read_csv(cleaned_path, parse_dates=["InvoiceDateTime"])
returns = pd.read_csv(returns_path) if os.path.exists(returns_path) else pd.DataFrame()

print (f"\nLoaded {len(df)} clealned sales rows")
df.head()

Looking in: c:\Users\Lenovo\OneDrive\Desktop\E-CommerceSales\output
Files here: ['audit_trail.csv', 'cleaned_sales.csv', 'eda_analysis.ipynb', 'excluded_rows.csv', 'non_product_transactions.csv', 'returns.csv']

Loaded 391150 clealned sales rows


,InvoiceNo,StockCode,Description,Quantity,InvoiceDateTime,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


**SECTION 1: HIGH LEVEL OVERVIEW**

In [10]:
print ("Total orders:       ", df["InvoiceNo"].nunique())
print ("Total customers:    ", df["CustomerID"].nunique())
print ("Total products:     ", df["StockCode"].nunique())
print ("Data range:       ", df["InvoiceDateTime"].min(), "to", df["InvoiceDateTime"].max())
print ("Total revenue:       ", f'{df["TotalPrice"].sum():,.2f}')

order_value = df.groupby("InvoiceNo")["TotalPrice"].sum()
print ("Order value range:  ", f'{order_value.mean():,.2f}')

Total orders:        18402
Total customers:     4334
Total products:      3659
Data range:        2010-12-01 08:26:00 to 2011-12-09 12:50:00
Total revenue:        8,737,227.64
Order value range:   474.80


**SECTION 2: REVENUE TRENDS OVERTIME**

In [11]:
df["year_month"] = df["InvoiceDateTime"].dt.to_period("M").astype(str)
monthly = df.groupby("year_month").agg(
    orders=("InvoiceNo", "nunique"),
    revenue=("TotalPrice", "sum")
).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly, x="year_month", y="revenue", color="#4C72B0")
plt.xticks(rotation=45, ha="right")
plt.title("Monthly Revenue")
plt.ylabel("Revenue")
plt.xlabel("Month")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/monthly_revenue.png", dpi=150)
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18684\2204048897.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
df["day_of_week"] = df["InvoiceDateTime"].dt.day_name()
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_revenue = df.groupby("day_of_week")["TotalPrice"].sum().reindex(dow_order)

plt.figure(figsize=(8, 5))
sns.barplot(x=dow_revenue.index, y=dow_revenue.values, color="#55A868")
plt.title("Revenue by Day of Week")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/revenue_by_dayofweek.png", dpi=150)
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18684\3659113144.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**SECTION 3: PRODUCT ANALYSIS**

In [14]:
top_products =(
    df.groupby(["StockCode", "Description"])
    .agg(revenue=("TotalPrice", "sum"), units_sold = ("Quantity", "sum"))
    .reset_index()
    .sort_values("revenue", ascending = False)
    .head(15)
)

plt.figure(figsize=(9, 6))
sns.barplot(data=top_products, y="Description", x="revenue", color="#C44E52")
plt.title("Top 15 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/top_products.png", dpi=150)
plt.show()

top_products

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18684\3368089615.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,StockCode,Description,revenue,units_sold
2599,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
1318,22423,REGENCY CAKESTAND 3 TIER,142264.75,12374
3456,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100392.10,36706
3441,85099B,JUMBO BAG RED RETROSPOT,85040.54,46078
2097,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916
2796,47566,PARTY BUNTING,68785.23,15279
3275,84879,ASSORTED COLOUR BIRD ORNAMENT,56413.03,35263
2003,23084,RABBIT NIGHT LIGHT,51251.24,27153
2957,79321,CHILLI LIGHTS,46265.11,9646
1001,22086,PAPER CHAIN KIT 50'S CHRISTMAS,42584.13,15591


**SECTION 4: GEOGRAPHY AND CUSTOMER ANALYSIS**

In [15]:
country_revenue = (
    df.groupby("Country")["TotalPrice"].sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(9, 6))
sns.barplot(x=country_revenue.values, y=country_revenue.index, color="#8172B2")
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/revenue_by_country.png", dpi=150)
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18684\4267586770.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
orders_per_customer = df.groupby("CustomerID")["InvoiceNo"].nunique()
repeat_pct = (orders_per_customer > 1).mean() * 100
print(f"Repeat customer rate: {repeat_pct:.1f}%")

plt.figure(figsize=(8, 5))
sns.histplot(orders_per_customer[orders_per_customer <= 20], bins=20, color="#4C72B0")
plt.title("Distribution of Orders per Customer (capped at 20)")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/orders_per_customer.png", dpi=150)
plt.show()

Repeat customer rate: 65.3%


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18684\155500338.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**SECTION  5: RFM FEATURE TABLE (RECENCY, FREQUENCY, MONETARY)**

In [17]:
snapshot_date = df["InvoiceDateTime"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDateTime", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum")
).reset_index()

rfm["Monetary"] = rfm["Monetary"].round(2)

rfm["R_Score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["RFM_Segment"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

def label_segment(row):
    if row["R_Score"] >= 3 and row["F_Score"] >= 3 and row["M_Score"] >= 3:
        return "Champions"
    if row["R_Score"] >= 3 and row["F_Score"] <= 2:
        return "New / Promising"
    if row["R_Score"] <= 2 and row["F_Score"] >= 3 and row["M_Score"] >= 3:
        return "At Risk (high value)"
    if row["R_Score"] <= 2 and row["F_Score"] <= 2:
        return "Lost / Dormant"
    return "Needs Attention"

rfm["Segment_Label"] = rfm.apply(label_segment, axis=1)

print(rfm["Segment_Label"].value_counts())

rfm_path = os.path.join(OUTPUT_DIR, "rfm_table.csv")
rfm.to_csv(rfm_path, index=False)
print(f"\nSaved: {rfm_path} ({len(rfm)} customers)")

rfm.head()

Segment_Label
Lost / Dormant          1509
Champions               1315
New / Promising          658
At Risk (high value)     450
Needs Attention          402
Name: count, dtype: int64

Saved: .\rfm_table.csv (4334 customers)


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,Segment_Label
0,12346,326,1,77183.60,1,1,4,114,Lost / Dormant
1,12347,2,7,4310.00,4,4,4,444,Champions
2,12348,75,4,1437.24,2,3,3,233,At Risk (high value)
3,12349,19,1,1457.55,3,1,3,313,New / Promising
4,12350,310,1,294.40,1,1,1,111,Lost / Dormant


**SECTION 6: RETURNS SNAPSHOTS**

In [18]:
if not returns.empty:
    cancellation_rate = returns["InvoiceNo"].nunique() / df["InvoiceNo"].nunique() * 100
    print(f"Cancellation rate: {cancellation_rate:.2f}%")
else:
    print("returns.csv not found - skipping cancellation rate")

returns.csv not found - skipping cancellation rate
